# Прогноз остатков физлиц в BYN на 30 дней

Для каждого выпуска берём **60 календарных дней** истории по дату выпуска включительно и прогнозируем **следующие 30 календарных дней**. Достаточно pandas и NumPy. Установите matplotlib для графиков. Для входа подготовьте DataFrame <code>df</code> с колонками <code>date</code> и <code>amount</code> (остаток в BYN) или укажите путь к CSV с этими колонками.

По умолчанию сравниваются последний факт, недельный наивный прогноз и недельный профиль с затухающим трендом. SARIMA и LSTM включаются отдельно. Все модели сравниваются на одних датах выпуска и горизонтах.

In [ ]:
import warnings

import numpy as np
import pandas as pd

CSV_PATH = None       # Например, "ostatki.csv". Если None, используется df.
RUN_SARIMA = False    # Для True нужен statsmodels.
RUN_LSTM = False      # Для True нужен PyTorch. Обучение на CPU.

LOOKBACK = 60
HORIZON = 30
VALID_DAYS = 60       # Максимальная длина периода выбора модели.
TEST_DAYS = 90        # Максимальная длина контрольного периода.
LIMIT_BYN = 30_000_000
MAX_STREAK = 7
SCALE = 1_000_000

NAIVE = "Последний факт"
WEEKLY = "Недельный наивный"
TREND = "Недельный профиль + тренд"
SARIMA = "SARIMA"
LSTM = "LSTM"
FIXED_LEADS = (1, 7, 14, HORIZON)
BUCKETS = ((1, 7), (8, 14), (15, HORIZON))
DAY = pd.Timedelta(days=1)

## Подготовка данных и ретропроверка

Строковые даты задавайте в формате YYYY-MM-DD. Пропущенные календарные даты сохраняются как неизвестные. При истории от 120 дней выделяем 30–60 дней на выбор модели, затем 30–90 дней на тест. При 90–119 днях строим только тестовое сравнение, при 60–89 днях доступен будущий прогноз. Без отдельного периода выбора итоговый прогноз строится по последнему факту.

Ретропроверка использует ежедневные выпуски, для которых все 30 целевых дат помещаются в свой период. К следующему выпуску становятся доступны факты, пришедшие к его дате. Для расчёта ошибок используются только известные целевые остатки.

In [ ]:
def prepare_daily(frame):
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("df должен быть pandas DataFrame.")
    if frame.empty or not frame.columns.is_unique or not {"date", "amount"} <= set(frame.columns):
        raise ValueError("df должен содержать строки и уникальные колонки date и amount.")

    date = frame["date"]
    if not pd.api.types.is_datetime64_any_dtype(date):
        iso_date = date.astype("string").str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}")
        if not iso_date.fillna(False).all():
            raise ValueError("Строковые даты должны иметь формат YYYY-MM-DD.")
        date = pd.to_datetime(date, format="%Y-%m-%d", errors="coerce")
    if date.isna().any() or date.dt.tz is not None or not date.eq(date.dt.normalize()).all():
        raise ValueError("Нужны существующие календарные даты без времени и часового пояса.")
    if date.duplicated().any():
        duplicates = date[date.duplicated()].dt.strftime("%Y-%m-%d").head().tolist()
        raise ValueError(f"Найдены повторяющиеся даты: {duplicates}.")

    amount = pd.to_numeric(frame["amount"], errors="coerce").to_numpy(dtype=float, na_value=np.nan)
    if not np.isfinite(amount).all():
        raise ValueError("В amount нужны конечные числовые остатки в BYN.")
    return pd.Series(amount, index=pd.DatetimeIndex(date, name="date"), name="amount").sort_index().asfreq("D")


if CSV_PATH is None and "df" not in globals():
    raise NameError("Подготовьте DataFrame df или укажите CSV_PATH в ячейке настроек.")

daily = prepare_daily(df if CSV_PATH is None else pd.read_csv(CSV_PATH))
if len(daily) < LOOKBACK:
    raise ValueError(f"Нужно минимум {LOOKBACK} календарных дней истории. Сейчас: {len(daily)}.")

In [ ]:
def holdout_origins(period):
    return period.index[:max(0, len(period) - HORIZON + 1)] - DAY


available_days = len(daily) - LOOKBACK
valid_days = min(VALID_DAYS, available_days // 2) if available_days >= 2 * HORIZON else 0
test_days = min(TEST_DAYS, available_days - valid_days) if available_days >= HORIZON else 0
valid = daily.iloc[len(daily) - test_days - valid_days:len(daily) - test_days]
test = daily.iloc[len(daily) - test_days:]
valid_start = valid.index[0] if valid_days else None
test_start = test.index[0] if test_days else None
valid_origins = holdout_origins(valid)
test_origins = holdout_origins(test)

split_info = pd.DataFrame.from_dict({
    name: {
        "начало": period.index.min(),
        "конец": period.index.max(),
        "календарных дней": len(period),
        "фактов": period.count(),
        "полных выпусков": len(origins),
    }
    for name, period, origins in (("valid", valid, valid_origins), ("test", test, test_origins))
}, orient="index")
display(split_info)

## Модели

- **Последний факт.** Повторяет последний известный остаток из окна.
- **Недельный наивный.** Для каждой будущей даты использует самый свежий факт за тот же день недели.
- **Недельный профиль + тренд.** Оценивает уровень, календарный день недели и линейный тренд. Регуляризация сдерживает подгонку на коротком окне, а вклад тренда постепенно затухает на горизонте.

При нехватке фактов недельная модель подставляет последний остаток для неизвестных дней недели. Модель с трендом использует недельный прогноз, если в окне меньше 14 фактов. SARIMA оценивается в каждом 60-дневном окне с недельным периодом 7, в случае несходимости использует недельную модель. Таблица после ретропроверки показывает число таких выпусков.

In [ ]:
def weekday_trend(history, dates):
    past_time = (history.index - dates[0] + DAY).days.to_numpy() / LOOKBACK
    future_time = np.cumsum(0.95 ** np.arange(len(dates))) / LOOKBACK

    def features(index, elapsed):
        weekdays = np.eye(7)[index.day_of_week]
        return np.column_stack([np.ones(len(index)), elapsed, weekdays])

    center = history.iloc[-1]
    design = features(history.index, past_time)
    target = (history.to_numpy() - center) / SCALE
    penalty = np.diag(np.sqrt([0, 1, 2, 2, 2, 2, 2, 2, 2]))
    coefficients = np.linalg.lstsq(
        np.vstack([design, penalty]), np.r_[target, np.zeros(design.shape[1])], rcond=None
    )[0]
    return center + SCALE * (features(dates, future_time) @ coefficients)


def sarima_forecast(window):
    center = window.dropna().iloc[-1]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        fitted = SARIMAX(
            (window - center) / SCALE,
            order=(1, 1, 0),
            seasonal_order=(1, 0, 0, 7),
        ).fit(disp=False, maxiter=35)
    values = fitted.forecast(HORIZON).to_numpy() * SCALE + center
    if not fitted.mle_retvals["converged"] or not np.isfinite(values).all():
        raise ValueError("SARIMA не сошлась.")
    return values


sarima_enabled = RUN_SARIMA
if RUN_SARIMA:
    try:
        from statsmodels.tsa.statespace.sarimax import SARIMAX
    except (ImportError, OSError) as error:
        sarima_enabled = False
        print(f"SARIMA пропущена. Проверьте установку statsmodels в ядре. {error}")


def local_predictions(daily, origins):
    names = [NAIVE, WEEKLY, TREND] + ([SARIMA] if sarima_enabled else [])
    predictions = {name: np.full((len(origins), HORIZON), np.nan) for name in names}
    fallbacks = dict.fromkeys(names, 0)
    for row, origin in enumerate(origins):
        window = daily.loc[origin - (LOOKBACK - 1) * DAY:origin]
        if len(window) != LOOKBACK or window.index[-1] != origin:
            raise ValueError("Для даты выпуска нужно полное 60-дневное календарное окно.")
        observed = window.dropna()
        if observed.empty:
            continue

        dates = pd.date_range(origin + DAY, periods=HORIZON)
        opening = observed.iloc[-1]
        predictions[NAIVE][row] = opening
        weekly = observed.groupby(observed.index.day_of_week).last()
        weekly = weekly.reindex(dates.day_of_week).fillna(opening).to_numpy()
        predictions[WEEKLY][row] = weekly
        if len(observed) >= 14:
            predictions[TREND][row] = weekday_trend(observed, dates)
        else:
            predictions[TREND][row] = weekly
            fallbacks[TREND] += 1

        if sarima_enabled:
            try:
                if len(observed) < 21 or observed.nunique() <= 1:
                    raise ValueError("Для оценки SARIMA мало изменяющихся остатков.")
                predictions[SARIMA][row] = sarima_forecast(window)
            except (ValueError, RuntimeError, np.linalg.LinAlgError):
                predictions[SARIMA][row] = weekly
                fallbacks[SARIMA] += 1
    return predictions, fallbacks

### Опциональная LSTM

Один слой LSTM получает относительные остатки и маску наблюдений за 60 дней и сразу выдаёт 30 прогнозов. Для valid и test модель обучается на фактах до начала соответствующего периода. Для будущего прогноза обучение использует окна до даты последнего факта включительно. Для работы модели нужно минимум 20 известных целей для каждого горизонта в обучающей части.

In [ ]:
def lstm_window_inputs(history):
    observed = np.isfinite(history)
    positions = np.where(observed, np.arange(history.shape[1]), -1)
    latest = np.maximum.accumulate(positions, axis=1)
    usable = observed.any(axis=1)
    opening = np.take_along_axis(history, latest[:, -1:].clip(min=0), axis=1)[:, 0]
    opening = np.where(usable, opening, 0.0)
    filled = np.take_along_axis(history, latest.clip(min=0), axis=1)
    filled = np.where(latest >= 0, filled, opening[:, None])
    relative = (filled - opening[:, None]) / SCALE
    return np.stack([relative, observed], axis=-1).astype(np.float32), opening, usable


def lstm_predictions(daily, origins, *, cutoff):
    import torch
    from torch import nn

    past = daily.loc[:cutoff].to_numpy()
    if len(past) < LOOKBACK + HORIZON:
        raise ValueError(f"Для LSTM нужно минимум {LOOKBACK + HORIZON} календарных дней до {cutoff.date()}.")
    windows = np.lib.stride_tricks.sliding_window_view(past, LOOKBACK + HORIZON)
    inputs, opening, usable = lstm_window_inputs(windows[:, :LOOKBACK])
    observed = np.isfinite(windows[:, LOOKBACK:])
    train_rows = usable & observed.any(axis=1)
    if (observed[train_rows].sum(axis=0) < 20).any():
        raise ValueError("Для LSTM нужно минимум 20 фактов на каждом горизонте обучения.")

    deltas_mln = (windows[train_rows, LOOKBACK:] - opening[train_rows, None]) / SCALE
    # Масштаб оцениваем только на известных таргетах обучения, до cutoff.
    change_scale = max(1.0, np.nanquantile(np.abs(deltas_mln), 0.75))
    inputs[:, :, 0] /= change_scale
    train_x = torch.from_numpy(inputs[train_rows])
    train_y = torch.from_numpy((deltas_mln / change_scale).astype(np.float32))
    train_mask = torch.from_numpy(observed[train_rows])

    class BalanceLSTM(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = nn.LSTM(input_size=2, hidden_size=24, batch_first=True)
            self.head = nn.Linear(24, HORIZON)

        def forward(self, history):
            _, (hidden, _) = self.encoder(history)
            return self.head(hidden[-1])

    positions = daily.index.get_indexer(origins)
    if (positions < LOOKBACK - 1).any():
        raise ValueError("У каждой даты выпуска LSTM должно быть полное 60-дневное входное окно.")
    all_history = np.lib.stride_tricks.sliding_window_view(daily.to_numpy(), LOOKBACK)
    forecast_x, forecast_opening, forecast_rows = lstm_window_inputs(all_history[positions - LOOKBACK + 1])
    forecast_x[:, :, 0] /= change_scale
    output = np.full((len(origins), HORIZON), np.nan)

    with torch.random.fork_rng(devices=[]):
        torch.default_generator.manual_seed(42)
        model = BalanceLSTM()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
        for _ in range(15):
            for batch in torch.randperm(len(train_x)).split(64):
                optimizer.zero_grad()
                predicted = model(train_x[batch])
                mask = train_mask[batch]
                loss = (predicted[mask] - train_y[batch][mask]).abs().mean()
                loss.backward()
                optimizer.step()

    model.eval()
    with torch.inference_mode():
        if forecast_rows.any():
            delta = model(torch.from_numpy(forecast_x[forecast_rows])).numpy().astype(float)
            output[forecast_rows] = forecast_opening[forecast_rows, None] + delta * SCALE * change_scale
    return output

In [ ]:
as_of = daily.index[-1]
origins_by_split = {
    "valid": valid_origins,
    "test": test_origins,
    "future": pd.DatetimeIndex([as_of]),
}
predictions_by_split = {}
fallbacks_by_split = {}
for split, origins in origins_by_split.items():
    predictions_by_split[split], fallbacks_by_split[split] = local_predictions(daily, origins)

if RUN_LSTM:
    cutoffs = {
        "valid": valid_start - DAY if valid_days else None,
        "test": test_start - DAY if test_days else None,
        "future": as_of,
    }
    lstm_by_split = {}
    try:
        for split, origins in origins_by_split.items():
            if len(origins):
                values = lstm_predictions(daily, origins, cutoff=cutoffs[split])
                available = np.isfinite(predictions_by_split[split][NAIVE])
                if not np.isfinite(values[available]).all():
                    raise ValueError("LSTM вернула неконечный прогноз.")
            else:
                values = np.empty((0, HORIZON))
            lstm_by_split[split] = values
    except (ImportError, OSError, ValueError, RuntimeError) as error:
        print(f"LSTM пропущена. {error}")
    else:
        for split, values in lstm_by_split.items():
            predictions_by_split[split][LSTM] = values

model_names = list(predictions_by_split["future"])
fallback_counts = pd.DataFrame(fallbacks_by_split).fillna(0).astype(int)
if fallback_counts.to_numpy().any():
    print("Выпуски с резервным прогнозом")
    display(fallback_counts.loc[fallback_counts.gt(0).any(axis=1)])

## Сравнение и выбор модели

MAE и RMSE измеряются в BYN. Положительное смещение означает, что прогноз завышает остаток. Точность сравнивается на общем наборе пар «дата выпуска — целевая дата» с известным фактом. Покрытие показывает долю таких пар среди всех запланированных.

Лимит допускает до 7 календарных дней подряд с абсолютной ошибкой свыше 30 млн BYN. Проверяем серии внутри каждого выпуска и отдельно по целевым датам на горизонтах 1, 7, 14 и 30 дней. Пропуски учитываются при оценке возможной длины серии. Из моделей с подтверждённым лимитом на valid выбираем модель с минимальной MAE. Если лимит на valid не подтверждён ни для одной модели, применяется выбор по MAE. На test все модели оцениваются с теми же настройками.

In [ ]:
def prediction_panel(daily, origins, predictions):
    leads = np.tile(np.arange(1, HORIZON + 1), len(origins))
    issue_dates = origins.repeat(HORIZON)
    dates = issue_dates + pd.to_timedelta(leads, unit="D")
    return pd.DataFrame({
        "origin": issue_dates,
        "date": dates,
        "lead": leads,
        "actual_byn": daily.reindex(dates).to_numpy(),
        **{name: values.ravel() for name, values in predictions.items()},
    })


def comparison_errors(panel):
    estimates = panel[model_names]
    errors = estimates.sub(panel["actual_byn"], axis=0).round(2)
    return errors.where(estimates.notna().all(axis=1), axis=0)


def score_table(errors):
    abs_error = errors.abs()
    mae = abs_error.mean()
    n_pairs = errors.count()
    scores = pd.DataFrame({
        "mae_byn": mae,
        "rmse_byn": np.sqrt(errors.pow(2).mean()),
        "bias_byn": errors.mean(),
        "n_pairs": n_pairs,
        "coverage_pct": 100 * n_pairs / len(errors) if len(errors) else np.nan,
        "limit_exceed_pct": 100 * abs_error.gt(LIMIT_BYN).sum() / n_pairs.replace(0, np.nan),
    }).rename_axis("модель")
    naive_mae = mae.loc[NAIVE]
    scores["skill_vs_naive_pct"] = 100 * (1 - mae / naive_mae) if naive_mae > 0 else np.nan
    return scores


valid_panel = prediction_panel(daily, valid_origins, predictions_by_split["valid"])
test_panel = prediction_panel(daily, test_origins, predictions_by_split["test"])
valid_errors = comparison_errors(valid_panel)
test_errors = comparison_errors(test_panel)

In [ ]:
LIMIT_STATUS_ORDER = ("failed", "insufficient_coverage", "insufficient_history", "passed")


def longest_run(mask):
    edges = np.flatnonzero(np.diff(np.r_[False, mask, False]))
    return int((edges[1::2] - edges[::2]).max(initial=0))


def worst_limit(statuses):
    return min(statuses, key=LIMIT_STATUS_ORDER.index, default="insufficient_history")


def check_limit(errors):
    missing = np.isnan(errors)
    exceeds = np.abs(errors) > LIMIT_BYN
    observed = longest_run(exceeds)
    possible = longest_run(exceeds | missing)
    if observed > MAX_STREAK:
        status = "failed"
    elif len(errors) <= MAX_STREAK:
        status = "insufficient_history"
    elif np.count_nonzero(~missing) <= MAX_STREAK or possible > MAX_STREAK:
        status = "insufficient_coverage"
    else:
        status = "passed"
    return {"status": status, "max_observed_days": observed, "max_possible_days": possible}


def limit_report(errors):
    # Каждая строка содержит 30 ошибок для одного календарного выпуска.
    errors = errors.to_numpy().reshape(-1, HORIZON)
    paths = [check_limit(path) for path in errors]
    rows = {
        "Траектории 1–30": {
            "status": worst_limit(check["status"] for check in paths),
            "max_observed_days": max((check["max_observed_days"] for check in paths), default=0),
            "max_possible_days": max((check["max_possible_days"] for check in paths), default=0),
        },
    }
    for lead in FIXED_LEADS:
        rows[f"Горизонт {lead}"] = check_limit(errors[:, lead - 1])
    return pd.DataFrame.from_dict(rows, orient="index").rename_axis("Проверка")



def evaluate(errors):
    limits = pd.concat({name: limit_report(errors[name]) for name in model_names},
                       names=["модель", "проверка"])
    scores = score_table(errors)
    checks = limits.groupby(level="модель")
    scores["limit_status"] = checks["status"].agg(worst_limit)
    scores["max_observed_days"] = checks["max_observed_days"].max()
    scores["max_possible_days"] = checks["max_possible_days"].max()
    return scores, limits


valid_metrics, valid_limits = evaluate(valid_errors)
test_metrics, test_limits = evaluate(test_errors)

selected_model = NAIVE
available_scores = valid_metrics.dropna(subset=["mae_byn"])
if available_scores.empty:
    print("На valid нет пар прогноз–факт. Используется последний факт.")
else:
    confirmed = available_scores.loc[available_scores["limit_status"].eq("passed")]
    selection_pool = confirmed if not confirmed.empty else available_scores
    selected_model = selection_pool["mae_byn"].idxmin()
    print("valid · выбор модели")
    display(valid_metrics.sort_values("mae_byn", kind="stable").round(2))

print(f"Модель для будущего прогноза — {selected_model}")
if test_days:
    print("test · контрольное сравнение")
    display(test_metrics.sort_values("mae_byn", kind="stable").round(2))
    display(test_limits["status"].unstack("проверка").reindex(model_names))
else:
    print("Для test нужно минимум 90 календарных дней истории, включая входное окно.")

In [ ]:
def scores_for_leads(panel, errors, groups):
    return pd.concat({
        label: score_table(errors.loc[panel["lead"].between(start, end)])
        for label, (start, end) in groups.items()
    }, names=["горизонт"])


metrics_by_range = scores_for_leads(test_panel, test_errors, {
    f"{start}–{end}": (start, end) for start, end in BUCKETS
})
metrics_by_lead = scores_for_leads(test_panel, test_errors, {
    lead: (lead, lead) for lead in range(1, HORIZON + 1)
})
if test_days:
    print("test · ошибки по диапазонам, дни")
    display(metrics_by_range.round(2))
    print("test · MAE в BYN на горизонтах 1, 7, 14 и 30 дней")
    display(metrics_by_lead["mae_byn"].unstack("модель").reindex(
        index=FIXED_LEADS, columns=model_names).round(2))

## Прогноз и ГЭПы

Дневной чистый поток приближаем изменением остатка к предыдущему дню. Накопленный ГЭП считаем от последнего фактического остатка. Для каждого интервала показываем самый большой спад от этой же точки как пиковую потребность в замещении средств. При наличии капитализации процентов и других безденежных изменений поток требует отдельной корректировки.

In [ ]:
future_dates = pd.date_range(as_of + DAY, periods=HORIZON, name="date")
future_comparison = pd.DataFrame(
    {name: values[0] for name, values in predictions_by_split["future"].items()},
    index=future_dates,
).round(2)

opening_byn = round(daily.at[as_of], 2)
forecast_df = future_comparison[selected_model].to_frame("forecast_byn")
forecast_df["lead"] = np.arange(1, HORIZON + 1)
forecast_df["net_flow_byn"] = np.diff(np.r_[opening_byn, forecast_df["forecast_byn"]]).round(2)
forecast_df["cumulative_gap_byn"] = (forecast_df["forecast_byn"] - opening_byn).round(2)
forecast_df["funding_need_byn"] = (-forecast_df["cumulative_gap_byn"]).clip(lower=0)

gaps = {}
for start, end in BUCKETS:
    segment = forecast_df.loc[forecast_df["lead"].between(start, end)]
    last = segment.iloc[-1]
    gaps[f"{start}–{end}"] = {
        "first_date": segment.index[0],
        "last_date": segment.index[-1],
        "closing_balance_byn": last["forecast_byn"],
        "net_flow_byn": round(segment["net_flow_byn"].sum(), 2),
        "cumulative_gap_byn": last["cumulative_gap_byn"],
        "peak_funding_byn": segment["funding_need_byn"].max(),
    }
gaps_df = pd.DataFrame.from_dict(gaps, orient="index").rename_axis("горизонт")

print(f"Лимит для модели «{selected_model}» · "
      f"valid: {valid_metrics.at[selected_model, 'limit_status']}, "
      f"test: {test_metrics.at[selected_model, 'limit_status']}")
display(forecast_df)
display(gaps_df)

## Графики

На тесте сравниваем MAE по горизонту и траектории из одного выпуска. На последнем графике — 60 дней истории и будущие прогнозы всех моделей.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Для построения графиков установите matplotlib в это ядро.")
else:
    if test_metrics["n_pairs"].gt(0).any():
        by_lead = metrics_by_lead["mae_byn"].unstack("модель").reindex(columns=model_names)
        ax = (by_lead / SCALE).plot(figsize=(11, 3.5), grid=True)
        ax.set(title="MAE на тесте", xlabel="Горизонт, календарные дни", ylabel="млн BYN", xticks=FIXED_LEADS)
        plt.show()

        fixed_origin = test_panel.loc[test_errors[NAIVE].notna(), "origin"].iloc[0]
        fixed_path = test_panel.loc[test_panel["origin"].eq(fixed_origin)].set_index("date")
        ax = (fixed_path[["actual_byn", *model_names]].rename(columns={"actual_byn": "Факт"}) / SCALE).plot(
            figsize=(11, 3.5), grid=True)
        ax.set(title=f"Тестовый выпуск от {fixed_origin.date()}", xlabel="Дата", ylabel="млн BYN")
        plt.show()

    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.plot(daily.index[-LOOKBACK:], daily.iloc[-LOOKBACK:] / SCALE, label="Факт", color="#242424")
    line_styles = ["-", "--", "-.", ":", (0, (5, 2, 1, 2))]
    for name, style in zip(model_names, line_styles):
        ax.plot(future_dates, future_comparison[name] / SCALE, label=name, linestyle=style,
                linewidth=2.4 if name == selected_model else 1.2)
    ax.axvline(as_of, color="#777777", linestyle="--", linewidth=.9)
    ax.set(title="Прогноз остатков на 30 календарных дней", ylabel="млн BYN", xlabel="Дата")
    ax.grid(alpha=.25)
    ax.legend()
    fig.autofmt_xdate()
    plt.show()